# Lambda = 4: Rerun of some VQE experiments


Notebook created by HLD for the work arXiv: 2503.13368 [quant-ph, hep-th]

In this notebook, we rerun some VQE experiments (with lower bounds) for bosonic SU(2) matrix model at $\Lambda = 4$ using different Estimator seeds. 

In [1]:
import numpy as np
import time
import matplotlib.pyplot as plt
from qiskit.circuit.library import TwoLocal, EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import NumPyEigensolver

import sys
sys.path.append('../../utility')
from vqe_run import *
from qc_ansatze import *

In [2]:
def load_H_txt(coupling):

    file = open(f"../../utility/pauliH_L4_g{coupling}.txt", "r")
    content = file.read()
    data = content.split(',\n')
    data[0] = data[0].split('\n')[1]
    data[-1] = data[-1].split('\n')[0]
    
    data_coeff = []
    data_str = []
    for i in range(len(data)):
        data_coeff.append(float(data[i].split('*')[0]))
        data_str.append(data[i].split('*')[1].split(' ')[1])
        
    Hpauli = list(zip(data_str,data_coeff))
    H4q = SparsePauliOp.from_list(Hpauli)
    
    solver = NumPyEigensolver(k=4)
    exact_solution = solver.compute_eigenvalues(H4q)
    #print("Exact Result of qubit hamiltonian:", np.real(exact_solution.eigenvalues))
    E_exact = np.round(np.real(exact_solution.eigenvalues)[0],5)
    print(f'Exact energy = {E_exact}')

    return H4q

In [3]:
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SPSA
from qiskit_algorithms.utils import algorithm_globals 
from qiskit_aer.primitives import Estimator as AerEstimator
import warnings
warnings.filterwarnings("ignore")

def run_qve_w_specified_optimizer(operator, optimizer, ansatz, seed = 170, iterations = 250):
    #storing values
    counts = []
    values = []
    def store_intermediate_result(eval_count, parameters, mean, std):
        counts.append(eval_count)
        values.append(mean)
        
    algorithm_globals.random_seed = seed
    noiseless_estimator = AerEstimator(
        run_options={"seed": seed, "shots": 1024},
        transpile_options={"seed_transpiler": seed},)
    opt = optimizer(maxiter = iterations)
    vqe = VQE(noiseless_estimator, ansatz, optimizer=opt, callback=store_intermediate_result)
    result = vqe.compute_minimum_eigenvalue(operator).eigenvalue.real
    print(f"VQE result: {result:.5f}")
    return result, values

In [7]:
ansatz_0a = TL_ansatz(12, 'ry', 'crx', "circular", 1)
ansatz_0d = TL_ansatz(12,['ry','y'], 'crx', 'circular', 1)
ansatz_1a = TL_ansatz(12, 'ry', 'crx', "full", 1)

Circuit ansatz with 36 parameters
Circuit ansatz with 36 parameters
Circuit ansatz with 90 parameters


In [11]:
ansatz_1c_su2 =  ef_ansatz(12, ['ry','rz'], "full", 1)

Circuit ansatz with 48 parameters


In [9]:
def vqe_rerun(seeds, H, optimizer, ansatz, iterations = 250):
    r_res=[]
    for i in range(len(seeds)):
        print(f'At step {i}, with {seeds[i]}')
        t0 = time.time()
        result, values = run_qve_w_specified_optimizer(H, optimizer, ansatz, seeds[i], iterations)
        t1 = time.time()
        print(f'Length of this optimization {len(values)}, time taken = {np.round(t1-t0,3)} \n')
        r_res.append(pd.DataFrame({f'seed_{seeds[i]}': values}))
    return r_res

# lambda = 0.2: effsu2_RyRz_f

In [13]:
H4q = load_H_txt(0.2)

Exact energy = 3.13406


In [18]:
seeds = [28]
r_res = vqe_rerun(seeds, H4q, SPSA, ansatz_1c_su2, iterations = 250)

At step 0, with 28
VQE result: 3.33804
Length of this optimization 551, time taken = 1199.106 



In [19]:
df0 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df0.to_csv('results_seeds/l4_l02_es2_RyRz_f_spsa_seeds.csv')

# lambda = 0.5: tl_Ry_c (COBYLA)

In [23]:
H4q = load_H_txt(0.5)

Exact energy = 3.29894


In [33]:
seeds = [28, 170, 225, 188, 18, 38, 48, 58]
r_res = vqe_rerun(seeds, H4q, COBYLA, ansatz_0a, iterations = 250)

At step 0, with 28
VQE result: 3.26214
Length of this optimization 250, time taken = 228.776 

At step 1, with 170
VQE result: 3.28859
Length of this optimization 250, time taken = 220.735 

At step 2, with 225
VQE result: 3.34911
Length of this optimization 250, time taken = 213.588 

At step 3, with 188
VQE result: 3.27436
Length of this optimization 250, time taken = 219.456 

At step 4, with 18
VQE result: 3.32138
Length of this optimization 250, time taken = 224.933 

At step 5, with 38
VQE result: 3.31995
Length of this optimization 250, time taken = 218.708 

At step 6, with 48
VQE result: 3.18170
Length of this optimization 250, time taken = 224.787 

At step 7, with 58
VQE result: 3.28802
Length of this optimization 250, time taken = 225.19 



In [36]:
df1 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df1.to_csv('results_seeds/l4_l05_tl_Ry_c_cobyla_seeds.csv')

In [37]:
seeds = [28, 170, 225, 188, 18, 38, 48, 58]
r_res = vqe_rerun(seeds, H4q, COBYLA, ansatz_0d, iterations = 250)

At step 0, with 28
VQE result: 3.36353
Length of this optimization 250, time taken = 250.994 

At step 1, with 170
VQE result: 3.25876
Length of this optimization 250, time taken = 252.39 

At step 2, with 225
VQE result: 4.54617
Length of this optimization 250, time taken = 249.473 

At step 3, with 188
VQE result: 3.34668
Length of this optimization 250, time taken = 246.865 

At step 4, with 18
VQE result: 3.48973
Length of this optimization 250, time taken = 251.962 

At step 5, with 38
VQE result: 3.31549
Length of this optimization 250, time taken = 246.694 

At step 6, with 48
VQE result: 3.48714
Length of this optimization 250, time taken = 252.807 

At step 7, with 58
VQE result: 3.21343
Length of this optimization 250, time taken = 251.675 



In [38]:
df2 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df2.to_csv('results_seeds/l4_l05_tl_RyY_c_cobyla_seeds.csv')

# lambda = 1.0: 

## tl_Ry_c (COBYLA)

In [13]:
H4q = load_H_txt(1.0)

Exact energy = 3.52625


In [42]:
seeds = [28, 170, 225, 188, 18, 38, 48, 58]
r_res = vqe_rerun(seeds, H4q, COBYLA, ansatz_0a, iterations = 250)

At step 0, with 28
VQE result: 3.51980
Length of this optimization 250, time taken = 234.371 

At step 1, with 170
VQE result: 4.45512
Length of this optimization 250, time taken = 223.502 

At step 2, with 225
VQE result: 3.59365
Length of this optimization 250, time taken = 223.622 

At step 3, with 188
VQE result: 3.52619
Length of this optimization 250, time taken = 215.798 

At step 4, with 18
VQE result: 3.79066
Length of this optimization 250, time taken = 218.452 

At step 5, with 38
VQE result: 5.07604
Length of this optimization 250, time taken = 217.49 

At step 6, with 48
VQE result: 3.73953
Length of this optimization 250, time taken = 220.363 

At step 7, with 58
VQE result: 3.43661
Length of this optimization 250, time taken = 212.424 



In [43]:
df3 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df3.to_csv('results_seeds/l4_l10_tl_Ry_c_cobyla_seeds.csv')

## tl_RyY_c (COBYLA)

In [51]:
seeds = [28, 170, 225, 188, 18, 38, 48, 58]
r_res = vqe_rerun(seeds, H4q, COBYLA, ansatz_0d, iterations = 250)

At step 0, with 28
VQE result: 3.72009
Length of this optimization 250, time taken = 239.632 

At step 1, with 170
VQE result: 3.54700
Length of this optimization 250, time taken = 250.897 

At step 2, with 225
VQE result: 3.89714
Length of this optimization 250, time taken = 243.682 

At step 3, with 188
VQE result: 3.33052
Length of this optimization 250, time taken = 245.189 

At step 4, with 18
VQE result: 3.81084
Length of this optimization 250, time taken = 254.901 

At step 5, with 38
VQE result: 3.58265
Length of this optimization 250, time taken = 243.269 

At step 6, with 48
VQE result: 3.27196
Length of this optimization 250, time taken = 251.246 

At step 7, with 58
VQE result: 3.41409
Length of this optimization 250, time taken = 241.19 



In [52]:
df4 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df4.to_csv('results_seeds/l4_l10_tl_RyY_c_cobyla_seeds.csv')

## tl_Ry_f (SPSA)

In [15]:
seeds = seeds = [28, 170, 225, 188, 18, 38, 48, 58]
r_res = vqe_rerun(seeds, H4q, SPSA, ansatz_1a, iterations = 350)

At step 0, with 28
VQE result: 3.48687
Length of this optimization 751, time taken = 1034.082 

At step 1, with 170
VQE result: 3.75099
Length of this optimization 751, time taken = 1069.237 

At step 2, with 225
VQE result: 3.63580
Length of this optimization 751, time taken = 1117.959 

At step 3, with 188
VQE result: 3.72211
Length of this optimization 751, time taken = 1127.641 

At step 4, with 18
VQE result: 3.53476
Length of this optimization 751, time taken = 1126.012 

At step 5, with 38
VQE result: 4.80874
Length of this optimization 751, time taken = 1110.816 

At step 6, with 48
VQE result: 3.39749
Length of this optimization 751, time taken = 1124.523 

At step 7, with 58
VQE result: 3.44197
Length of this optimization 751, time taken = 1056.534 



In [16]:
df5 = pd.concat([r_res[i] for i in range(len(r_res))], axis = 1)
df5.to_csv('results_seeds/l4_l10_tl_Ry_f_spsa_seeds.csv')
df5

,seed_28,seed_170,seed_225,seed_188,seed_18,seed_38,seed_48,seed_58
0,16.859608,20.894629,23.325423,22.687451,25.584049,25.259189,13.602385,14.326587
1,16.353516,23.290311,20.342914,25.870057,25.710744,20.234078,15.779348,12.389893
2,17.435869,21.824616,22.147461,24.620030,23.929365,20.748314,13.460314,13.604907
3,15.362499,23.191825,21.395506,24.793853,26.616129,24.217845,16.061729,12.632378
4,16.483919,21.255647,20.728281,23.593941,26.304178,18.865341,14.980502,11.790861
...,...,...,...,...,...,...,...,...
746,3.520296,3.973799,3.353024,3.879339,3.682184,4.782505,3.580081,3.419599
747,3.737148,3.787710,3.529423,4.123642,3.726450,4.601043,3.434094,3.563324
748,3.449138,4.148593,3.628183,3.624382,3.660712,4.598428,3.677389,3.790029
749,3.869318,3.993691,3.924423,3.861519,3.680043,4.942288,3.536757,3.894534
